# Deployment

Now, we'll cover how to deploy our LangGraph agent as a **Databricks Model Serving endpoint** — packaging the graph with MLflow and serving it via the same OpenAI-compatible API we've been using throughout the course.

## Concepts

The deployment approach uses:

**MLflow** —
- Tracks experiments and packages models as reusable artifacts
- The `mlflow.langchain` flavour natively supports LangGraph graphs
- Logs the graph as an MLflow model with all dependencies

**Databricks Model Serving** —
- Hosts MLflow models behind a REST endpoint
- Provides an OpenAI-compatible chat API (`/serving-endpoints/<name>/invocations`)
- Scales automatically, supports GPU, and integrates with workspace auth

**Workflow**:
1. Define the LangGraph agent (same as notebook 5/6)
2. Log it to MLflow with `mlflow.langchain.log_model()`
3. Register the model in Unity Catalog (or the workspace registry)
4. Create a Model Serving endpoint pointing at that registered model
5. Invoke the endpoint with `openai.OpenAI` or `ChatOpenAI` — just like the rest of our course

## Step 1 — Define the Agent

We define a **CS4603 study assistant** agent with four tools:

| Tool | What it does |
|------|-------------|
| `calculate` | Math operations (add, subtract, multiply, divide, power) |
| `convert_temperature` | Convert between °C, °F, and K |
| `analyze_text` | Word count, sentence count, avg word length, reading time |
| `lookup_cs4603_topic` | Look up course topics (tokens, RAG, agents, etc.) |

The agent uses a system prompt and `tools_condition` for routing.

In [ ]:
%run ../../langchain_common.py

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage, SystemMessage


def calculate(a: float, b: float, operation: str) -> str:
    """Perform a math calculation on two numbers.

    Args:
        a: first number
        b: second number
        operation: one of 'add', 'subtract', 'multiply', 'divide', 'power'
    """
    ops = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: "Error: division by zero" if y == 0 else x / y,
        "power": lambda x, y: x ** y,
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Use: {', '.join(ops)}"
    result = ops[operation](a, b)
    return f"{a} {operation} {b} = {result}"


def convert_temperature(value: float, from_unit: str, to_unit: str) -> str:
    """Convert a temperature between Celsius, Fahrenheit, and Kelvin.

    Args:
        value: the temperature value to convert
        from_unit: source unit — 'C', 'F', or 'K'
        to_unit: target unit — 'C', 'F', or 'K'
    """
    if from_unit == "C":
        celsius = value
    elif from_unit == "F":
        celsius = (value - 32) * 5 / 9
    elif from_unit == "K":
        celsius = value - 273.15
    else:
        return f"Unknown unit '{from_unit}'. Use C, F, or K."

    if to_unit == "C":
        result = celsius
    elif to_unit == "F":
        result = celsius * 9 / 5 + 32
    elif to_unit == "K":
        result = celsius + 273.15
    else:
        return f"Unknown unit '{to_unit}'. Use C, F, or K."

    return f"{value}°{from_unit} = {result:.2f}°{to_unit}"


def analyze_text(text: str) -> str:
    """Analyze text and return statistics: word count, sentence count,
    average word length, and estimated reading time.

    Args:
        text: the text to analyze
    """
    words = text.split()
    word_count = len(words)
    sentence_count = sum(1 for c in text if c in ".!?") or 1
    avg_word_len = sum(len(w.strip(".,!?;:")) for w in words) / max(word_count, 1)
    reading_time_sec = word_count / 3.5
    return (
        f"Words: {word_count} | Sentences: {sentence_count} | "
        f"Avg word length: {avg_word_len:.1f} chars | Reading time: {reading_time_sec:.0f}s"
    )


def lookup_cs4603_topic(topic: str) -> str:
    """Look up a CS4603 course topic and return a brief summary.

    Args:
        topic: keyword to search for, e.g. 'tokens', 'embeddings', 'RAG',
               'agents', 'langgraph', 'tool calling', 'prompt engineering'
    """
    knowledge = {
        "tokens": "Tokens are sub-word units the LLM processes. Use tiktoken to count them.",
        "embeddings": "Embeddings map text to dense vectors for semantic similarity and retrieval.",
        "rag": "Retrieval-Augmented Generation: embed docs, retrieve relevant chunks, feed to LLM.",
        "agents": "Agents are LLMs that decide which tools to call using the ReAct pattern.",
        "langgraph": "LangGraph models agent workflows as directed graphs with nodes, edges, and state.",
        "tool calling": "Tool calling lets the LLM output structured function calls instead of text.",
        "prompt engineering": "System messages, few-shot examples, chain-of-thought, temperature control.",
        "mlflow": "MLflow tracks experiments, logs models, and serves them via Databricks endpoints.",
    }
    key = topic.lower().strip()
    matches = [v for k, v in knowledge.items() if key in k or k in key]
    if matches:
        return matches[0]
    return f"Topic '{topic}' not found. Available: {', '.join(knowledge)}"


tools = [calculate, convert_temperature, analyze_text, lookup_cs4603_topic]
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = (
    "You are a helpful CS4603 study assistant. You can perform calculations, "
    "convert temperatures, analyze text, and look up course topics about LLMs "
    "and LangChain. Use the available tools when appropriate."
)

In [ ]:
def assistant(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

graph = builder.compile()
print("Graph compiled successfully")

{'assistant_id': '228f9934-0cdd-5383-92c8-ee8422522cc2',
 'graph_id': 'router',
 'config': {},
 'context': {},
 'metadata': {'created_by': 'system'},
 'name': 'router',
 'created_at': '2026-04-16T15:43:58.055650+00:00',
 'updated_at': '2026-04-16T15:43:58.055650+00:00',
 'version': 1,
 'description': None}

In [ ]:
# Quick sanity check — try each tool type
for query in [
    "What is 2 to the power of 10?",
    "Convert 100 degrees Fahrenheit to Celsius",
    "What is RAG in the context of LLMs?",
]:
    print(f"\n>>> {query}")
    result = graph.invoke({"messages": [HumanMessage(content=query)]})
    result["messages"][-1].pretty_print()

## Step 2 — Log the Agent to MLflow

We use `mlflow.langchain.log_model()` to package the compiled graph. MLflow will:
- Capture the graph definition and all its dependencies
- Store it as a versioned artifact we can later register and serve

In [ ]:
import mlflow

mlflow.set_experiment("wk5-deployment")

with mlflow.start_run(run_name="langgraph-agent") as run:
    model_info = mlflow.langchain.log_model(
        lc_model=graph,
        artifact_path="langgraph_agent",
        input_example={"messages": [{"role": "user", "content": "Add 2 and 3."}]},
    )
    print(f"Model logged: {model_info.model_uri}")
    print(f"Run ID: {run.info.run_id}")

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '9c5db9dc-7f45-490b-93de-c6c4e044fe80'}
{'content': '', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 18, 'prompt_tokens': 59, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DVJDjULwae7DpCphlaqeVtR6S2vwT', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': 

## Step 3 — Load and Test the Logged Model Locally

Before deploying, we reload the model from MLflow and verify it produces correct output.

In [ ]:
loaded_model = mlflow.langchain.load_model(model_info.model_uri)
result = loaded_model.invoke({"messages": [HumanMessage(content="What is 5 + 7?")]})
for msg in result["messages"]:
    msg.pretty_print()

## Step 4 — Register the Model

Register the logged model in the MLflow Model Registry. On Databricks this would target Unity Catalog (e.g. `catalog.schema.model_name`). Locally we use the workspace registry.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
model_name = "langgraph_agent"

# Register the model (creates the registered model if it doesn't exist)
mv = client.create_model_version(
    name=model_name,
    source=model_info.model_uri,
    run_id=run.info.run_id,
)
print(f"Registered model '{model_name}' version {mv.version}")

## Step 5 — Deploy to Databricks Model Serving

On Databricks, you would create a **Model Serving endpoint** pointing at the registered model. The cell below shows the API call to create the endpoint.

> **Note**: This requires running in a Databricks workspace with appropriate permissions. The endpoint will be available at `{DATABRICKS_HOST}/serving-endpoints/langgraph-agent/invocations`.

```python
# Run this in a Databricks notebook or with workspace-level permissions:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

w.serving_endpoints.create(
    name="langgraph-agent",
    config={
        "served_entities": [{
            "entity_name": "catalog.schema.langgraph_agent",  # Unity Catalog path
            "entity_version": "1",
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
        }]
    }
)
```

Once the endpoint is ready (status = `READY`), you can call it just like any other Databricks model serving endpoint.

## Step 6 — Call the Deployed Endpoint

Once the serving endpoint is live, we interact with it using the same OpenAI-compatible client pattern we've used throughout the course.

In [ ]:
import openai

# Point at the deployed endpoint (same pattern as all our other notebooks)
serving_client = openai.OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

response = serving_client.chat.completions.create(
    model="cs4603-langgraph-agent",  # name of your serving endpoint
    messages=[{"role": "user", "content": "Convert 0 Kelvin to Fahrenheit and explain what that temperature means."}],
)
print(response.choices[0].message.content)

## Summary

| Step | Action |
|------|--------|
| 1 | Define CS4603 study assistant (4 tools: calculate, temperature, text stats, topic lookup) |
| 2 | Log to MLflow with `mlflow.langchain.log_model()` |
| 3 | Verify locally by reloading the model |
| 4 | Register in Model Registry / Unity Catalog |
| 5 | Create a Databricks Model Serving endpoint |
| 6 | Call the endpoint via OpenAI-compatible API |